In [ ]:
# -*- coding: utf-8 -*-
"""
IEEE.py  -  StampedeGuard (YOLOv8 + DeepSORT + dynamic crowd graph + ST-GNN + CIS + XAI)
=========================================================================================
Google Colab friendly. Run it either way:
    !python IEEE.py            (in a Colab cell)      or      %run IEEE.py

WHAT THIS DOES (the same pipeline your paper describes)
  1. Gets the Kaggle "crowd-detection-video" videos (auto download, or upload them to /content)
  2. YOLOv8 person detection  ->  DeepSORT tracking
  3. Per-person features (position, speed, acceleration, heading) from finite differences
  4. A dynamic crowd graph at every time step (nodes = people, edges = nearby people)
  5. A TRAINED spatio-temporal GNN that predicts crowd risk HORIZON_S seconds ahead
  6. Crowd Instability Score (CIS) = 0.40*density + 0.35*proximity + 0.25*movement
  7. Gradient-based explanation (which people / links / factors drove the prediction)
  8. Honest evaluation: leave-one-VIDEO-out cross-validation, ablations, baseline
  9. All paper figures, annotated videos, metrics and ready-to-paste paper text

IMPORTANT HONESTY NOTES (please read)
  * Metrics are whatever the evaluation produces. Nothing is hard-coded or tuned to hit a target.
  * The labels below are VIDEO-LEVEL labels copied from your old notebook. I have not seen the
    videos. Watch each video and correct VIDEO_LABELS (or add SEGMENT_LABELS) before you publish.
  * With only 7 videos the results are a proof of concept, not a deployment-grade claim.
"""

# %% ------------------------------------------------------------------------------------------
# 0. INSTALL (only what is missing)
# ---------------------------------------------------------------------------------------------
import importlib, os, sys, subprocess


def _ensure(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
    except Exception:
        print(f"Installing {pip_name or import_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


SYNTH = os.environ.get("SG_SYNTH", "0") == "1"      # internal self-test with synthetic tracks
for _imp, _pip in [("torch", "torch"), ("cv2", "opencv-python"), ("sklearn", "scikit-learn"),
                   ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    _ensure(_imp, _pip)
if not SYNTH:
    _ensure("ultralytics", "ultralytics")
    _ensure("deep_sort_realtime", "deep-sort-realtime")
    _ensure("kagglehub", "kagglehub")
    try:
        import pkg_resources  # noqa  (deep-sort-realtime needs it; newer setuptools removed it)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"])

import re, json, time, math, pickle, random, shutil, glob, builtins, functools
print = functools.partial(builtins.print, flush=True)      # show every line immediately (no buffering)
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef)

# %% ------------------------------------------------------------------------------------------
# 1. CONFIGURATION  (everything you may want to change is here)
# ---------------------------------------------------------------------------------------------
CFG = dict(
    KAGGLE_DATASET="ubaydulloasatullaev/crowd-detection-video",
    VIDEO_DIR=None,              # folder containing the videos; None = auto (/content or Kaggle)
    YOLO_WEIGHTS="yolov8s.pt",   # yolov8n.pt = faster, yolov8m.pt = more accurate
    YOLO_CONF=0.25,
    MAX_WIDTH=1920,              # big (4K) frames are shrunk to this width before analysis
    TILE_ROWS=2, TILE_COLS=2,    # TILED DETECTION: frame is also scanned in overlapping tiles so small, far-away
    TILE_OVERLAP=0.25,           #   people are seen at higher effective resolution (1x1 = off, much faster)
    YOLO_IMGSZ=960,              # larger image size helps small people in crowds
    TARGET_DT=0.20,              # seconds between analysed frames (sets the frame stride)
    MAX_STEPS=800,               # max analysed frames per video (800 * 0.2 s = 160 s)
    K_HISTORY=10,                # number of past graphs fed to the ST-GNN (10 * 0.2 s = 2 s)
    HORIZON_S=3.0,               # prediction horizon in seconds (your clips are short: 6-40 s)
    R_FACTOR=1.0,                # interaction radius r = R_FACTOR * average person height (pixels)
    MAX_NODES=100,               # cap on people per graph (random subset above this)
    DEG_REF=6.0,                 # neighbours regarded as "fully dense" (density component)
    V_REF=2.0,                   # body-heights / second regarded as "fast"
    N_REF=40.0,                  # crowd size regarded as "full"
    CIS_WEIGHTS=(0.40, 0.35, 0.25),   # density, proximity, movement (as in the paper)
    CIS_THRESHOLD=0.50,          # CIS band boundary (paper Table II)
    HIDDEN=32, EPOCHS=30, BATCH=32, LR=1e-3, WEIGHT_DECAY=1e-4, PATIENCE=8,
    WIN_STRIDE_TRAIN=2, VAL_FRAC=0.20, SEED=42,
    RUN_ABLATIONS=True,          # also train "no graph" and "no temporal" variants
    WRITE_VIDEOS=True,           # write annotated mp4 for every video
    FORCE_RETRACK=True,          # True = ignore cached YOLO/DeepSORT results (set False after a good run)
)

if SYNTH:                                                    # quick self-test settings
    CFG.update(EPOCHS=4, WRITE_VIDEOS=True, MAX_NODES=40, PATIENCE=3)

# LABELS (0 = stable / calm crowd, 1 = unstable / risky crowd).  Supervised metrics need ground truth,
# and only YOU can judge your videos. Two ways to give it (no code editing needed for the first):
#   (1) put a file  labels.csv  in /content :   video,label   (one line per video, e.g.  video1,0)
#   (2) otherwise the labels from your old notebook for video1..video7 are used (CHECK THEM).
VIDEO_LABELS = {"video1": 0, "video2": 1, "video3": 0, "video4": 1, "video5": 1, "video6": 0, "video7": 1}
LABEL_SOURCE = "built-in labels from your old notebook (video1-video7)"
for _p in ["/content/labels.csv", "labels.csv"]:
    if os.path.exists(_p) and not SYNTH:
        _df = pd.read_csv(_p); _df.columns = [c.strip().lower() for c in _df.columns]
        VIDEO_LABELS = {str(r["video"]).strip().lower().replace(".mp4", ""): int(r["label"]) for _, r in _df.iterrows()}
        LABEL_SOURCE = f"labels file {_p}"
        break
if SYNTH:
    VIDEO_LABELS = {"video1": 0, "video2": 1, "video3": 0, "video4": 1, "video5": 1, "video6": 0, "video7": 1}

# Optional, MUCH better: label only the risky time ranges  {video: [(start_s, end_s, label), ...]}
# Example: SEGMENT_LABELS = {"video2": [(0, 40, 0), (40, 90, 1)]}
SEGMENT_LABELS = {}

WORK = "/content/stampedeguard" if os.path.isdir("/content") else os.path.abspath("./stampedeguard_out")
for sub in ["cache", "figures", "videos", "results"]:
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


if DEVICE == "cpu" and not SYNTH:      # CPU would take hours with the accurate settings -> automatic FAST MODE
    CFG.update(YOLO_WEIGHTS="yolov8n.pt", YOLO_IMGSZ=640, TILE_ROWS=1, TILE_COLS=1, MAX_WIDTH=1280, MAX_STEPS=150, EPOCHS=15)
    print("[FAST MODE] No GPU found, so lighter settings are used (yolov8n, no tiling, 150 frames/video).")
    print("            For the best accuracy: Runtime > Change runtime type > T4 GPU (then re-upload the files and rerun).")
seed_all(CFG["SEED"])
print("=" * 70)
print("STAMPEDEGUARD  |  device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only (use Runtime > GPU)")
print("Output folder:", WORK)
print("=" * 70)

# %% ------------------------------------------------------------------------------------------
# 2. GET THE VIDEOS
# ---------------------------------------------------------------------------------------------
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".webm")


def video_key(path):
    """'pexels_videos_2740 (1080p).mp4' -> 'pexels_videos_2740' ; 'video1.mp4.mp4' -> 'video1'"""
    name = os.path.basename(path).lower()
    m = re.match(r"(video)[\s_\-]*(\d+)", name)
    if m:
        return f"video{int(m.group(2))}"
    s = os.path.splitext(name)[0]
    s = re.sub(r"\(\s*\d{3,4}p\s*\)", "", s)      # (1080p)
    s = re.sub(r"\(\s*\d+\s*\)", "", s)            # (1)
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


def natural(path):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", os.path.basename(path).lower())]


def scan_videos(folder):
    found = []
    for root, _, files in os.walk(folder):
        if any(p in root for p in ["stampedeguard", "/outputs", "/sample_data"]):
            continue
        for f in files:
            if f.lower().endswith(VIDEO_EXT) and "_annotated" not in f.lower():
                found.append(os.path.join(root, f))
    return sorted(found, key=natural)


def get_videos():
    if CFG["VIDEO_DIR"]:
        vids = scan_videos(CFG["VIDEO_DIR"])
        if vids:
            return vids
    vids = scan_videos("/content") if os.path.isdir("/content") else []
    if vids:
        print("Using videos already in /content")
        return vids
    try:
        import kagglehub
        print("Downloading dataset from Kaggle:", CFG["KAGGLE_DATASET"])
        path = kagglehub.dataset_download(CFG["KAGGLE_DATASET"])
        vids = scan_videos(path)
        if vids:
            return vids
    except Exception as e:
        print("Kaggle download failed:", repr(e))
    raise FileNotFoundError(
        "No videos found. Either (a) upload your 7 videos to /content (left sidebar > Files), or "
        "(b) set Kaggle credentials (Colab: Secrets KAGGLE_USERNAME / KAGGLE_KEY) and re-run.")


# %% ------------------------------------------------------------------------------------------
# 3. YOLOv8 + DeepSORT  ->  tracks (cached)
# ---------------------------------------------------------------------------------------------

def detect_people(yolo, frame):
    """Full-frame pass (large people) + overlapping tiles (small / distant people), merged with NMS."""
    H, W = frame.shape[:2]; R, C, ov = CFG["TILE_ROWS"], CFG["TILE_COLS"], CFG["TILE_OVERLAP"]
    B, S = [], []

    def run(img, ox, oy):
        res = yolo.predict(img, conf=CFG["YOLO_CONF"], classes=[0], imgsz=CFG["YOLO_IMGSZ"], verbose=False, device=DEVICE)[0]
        if res.boxes is not None and len(res.boxes):
            b = res.boxes.xyxy.cpu().numpy().copy(); b[:, [0, 2]] += ox; b[:, [1, 3]] += oy
            B.append(b); S.append(res.boxes.conf.cpu().numpy())

    run(frame, 0, 0)
    if R * C > 1:
        th, tw = int(H / R * (1 + ov)), int(W / C * (1 + ov))
        for r in range(R):
            for c in range(C):
                y0 = int(min(max(0, r * H / R - ov * H / R / 2), H - th)); x0 = int(min(max(0, c * W / C - ov * W / C / 2), W - tw))
                run(frame[y0:y0 + th, x0:x0 + tw], x0, y0)
    if not B:
        return np.zeros((0, 4), np.float32), np.zeros(0, np.float32)
    b, s = np.concatenate(B), np.concatenate(S)
    import torchvision
    keep = torchvision.ops.nms(torch.from_numpy(b).float(), torch.from_numpy(s).float(), 0.5).numpy()
    return b[keep], s[keep]


def track_video(path, key, yolo=None):
    cache = os.path.join(WORK, "cache", f"{key}_tracks.pkl")
    if os.path.exists(cache) and not CFG["FORCE_RETRACK"]:
        with open(cache, "rb") as f:
            return pickle.load(f)
    from deep_sort_realtime.deepsort_tracker import DeepSort
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    W0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H0 = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    sc = min(1.0, CFG["MAX_WIDTH"] / W0); W, H = int(W0 * sc), int(H0 * sc)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    stride = max(1, int(round(fps * CFG["TARGET_DT"])))
    tracker = DeepSort(max_age=8, n_init=2, max_iou_distance=0.7, max_cosine_distance=0.3,
                       nn_budget=100, embedder="mobilenet", half=(DEVICE == "cuda"),
                       embedder_gpu=(DEVICE == "cuda"))
    steps, t_yolo, t_track, idx = [], 0.0, 0.0, 0
    print(f"  {key}: {W}x{H} @ {fps:.1f} fps, {total} frames, stride {stride}")
    est = max(1, min(CFG["MAX_STEPS"], total // stride)); t_begin = time.time()
    print(f"    {key}: starting detection of ~{est} frames (the first frame is slow: models warm up) ...")
    while len(steps) < CFG["MAX_STEPS"]:
        if not cap.grab():
            break
        if idx % stride == 0:
            ok, frame = cap.retrieve()
            if not ok:
                break
            if sc < 1.0:
                frame = cv2.resize(frame, (W, H))
            t0 = time.time()
            bxs, cfs = detect_people(yolo, frame)
            t_yolo += time.time() - t0
            dets = [([float(x1), float(y1), float(x2 - x1), float(y2 - y1)], float(c), "person") for (x1, y1, x2, y2), c in zip(bxs, cfs)]
            t0 = time.time()
            tracks = tracker.update_tracks(dets, frame=frame)
            t_track += time.time() - t0
            ids, boxes = [], []
            for t in tracks:
                if t.is_confirmed() and t.time_since_update == 0:
                    ids.append(int(t.track_id)); boxes.append(t.to_ltrb())
            if len(steps) % 20 == 0:
                el = time.time() - t_begin; done = max(1, len(steps))
                print(f"    {key}: frame {done}/{est}  elapsed {el:.0f}s  ~{el / done * (est - done):.0f}s left", flush=True)
            steps.append(dict(frame=idx, t=idx / fps, ids=np.array(ids, dtype=np.int64),
                              boxes=np.array(boxes, dtype=np.float32).reshape(-1, 4)))
        idx += 1
    cap.release()
    n = max(1, len(steps))
    out = dict(key=key, path=path, fps=fps, W=W, H=H, dt=stride / fps, stride=stride, total_frames=total,
               steps=steps, ms_yolo=1000 * t_yolo / n, ms_track=1000 * t_track / n)
    with open(cache, "wb") as f:
        pickle.dump(out, f)
    return out


def synthetic_tracks(key, label, seed):
    """Self-test only (SG_SYNTH=1): fake crowd with calm (0) or chaotic (1) behaviour."""
    rng = np.random.RandomState(seed)
    W, H, fps, T = 1280, 720, 25.0, 320
    n = 18 if label == 0 else 38
    pos = rng.rand(n, 2) * [W, H]
    base = rng.randn(2); base /= np.linalg.norm(base)
    vel = np.tile(base * 18, (n, 1)) + rng.randn(n, 2) * (2 if label == 0 else 14)
    steps = []
    for t in range(T):
        if label == 1 and t % 40 == 0:
            vel = rng.randn(n, 2) * 30
        pos += vel * 0.25 + rng.randn(n, 2) * (1 if label == 0 else 6)
        pos[:, 0] %= W; pos[:, 1] %= H
        hh = 60 + 0.05 * pos[:, 1]
        boxes = np.stack([pos[:, 0] - hh / 4, pos[:, 1] - hh / 2, pos[:, 0] + hh / 4, pos[:, 1] + hh / 2], 1)
        kk = np.sort(rng.choice(n, rng.randint(2, n + 1), replace=False))
        steps.append(dict(frame=int(t * 6), t=t * 6 / fps, ids=np.arange(n)[kk], boxes=boxes[kk].astype(np.float32)))
    return dict(key=key, path=None, fps=fps, W=W, H=H, dt=6 / fps, stride=6, total_frames=T * 6,
                steps=steps, ms_yolo=0.0, ms_track=0.0)


# %% ------------------------------------------------------------------------------------------
# 4. FEATURES  ->  dynamic crowd graph + CIS components
# ---------------------------------------------------------------------------------------------
NODE_F = 6                                   # [x, y, speed, acceleration, cos(theta), sin(theta)]
GLOB_F = 6                                   # [C_d, C_p, C_m, CIS, crowd size, mean speed]
NODE_NAMES = ["Position X", "Position Y", "Speed", "Acceleration", "Heading (cos)", "Heading (sin)"]
GLOB_NAMES = ["Density component", "Proximity component", "Movement component",
              "Instantaneous CIS", "Crowd size", "Mean speed"]


def build_features(tr):
    """finite differences: V=(P_t-P_{t-1})/dt, A=(V_t-V_{t-1})/dt, theta=atan2(vy,vx)."""
    W, H, dt = tr["W"], tr["H"], tr["dt"]
    wd, wp, wm = CFG["CIS_WEIGHTS"]
    prev, rng, feats = {}, np.random.RandomState(0), []
    for st in tr["steps"]:
        ids, b = st["ids"], st["boxes"]
        n = len(ids)
        if n > CFG["MAX_NODES"]:
            keep = np.sort(rng.choice(n, CFG["MAX_NODES"], replace=False)); ids, b = ids[keep], b[keep]; n = len(ids)
        if n == 0:
            feats.append(dict(t=st["t"], frame=st["frame"], ids=ids, pos=np.zeros((0, 2), np.float32),
                              X=np.zeros((0, NODE_F), np.float32), A=np.zeros((0, 0), np.float16),
                              G=np.zeros(GLOB_F, np.float32), Cd=0., Cp=0., Cm=0., CIS=0., n=0, deg=np.zeros(0)))
            continue
        cx = (b[:, 0] + b[:, 2]) / 2; cy = (b[:, 1] + b[:, 3]) / 2
        h = np.maximum(b[:, 3] - b[:, 1], 8.0)
        vx = np.zeros(n); vy = np.zeros(n); ax = np.zeros(n); ay = np.zeros(n)
        for i, pid in enumerate(ids):
            if pid in prev:
                px, py, pvx, pvy = prev[pid]
                rvx, rvy = (cx[i] - px) / dt, (cy[i] - py) / dt
                vx[i] = 0.6 * rvx + 0.4 * pvx; vy[i] = 0.6 * rvy + 0.4 * pvy      # light smoothing
                ax[i], ay[i] = (vx[i] - pvx) / dt, (vy[i] - pvy) / dt
            prev[pid] = (cx[i], cy[i], vx[i], vy[i])
        speed = np.hypot(vx, vy) / h                       # body-heights per second (perspective aware)
        acc = np.hypot(ax, ay) / h
        theta = np.arctan2(vy, vx)
        moving = speed > 0.3
        uh = np.stack([np.cos(theta), np.sin(theta)], 1) * moving[:, None]
        X = np.stack([cx / W, cy / H, np.clip(speed / CFG["V_REF"], 0, 3), np.clip(acc / 4.0, 0, 3),
                      uh[:, 0], uh[:, 1]], 1).astype(np.float32)
        # ---- graph: edge if distance < r_ij (average body height of the pair * R_FACTOR)
        P = np.stack([cx, cy], 1)
        D = np.linalg.norm(P[:, None] - P[None], axis=-1)
        Rr = CFG["R_FACTOR"] * (h[:, None] + h[None]) / 2
        adj = (D < Rr) & ~np.eye(n, dtype=bool)
        dirsim = np.where(moving[:, None] & moving[None], 0.5 + 0.5 * (uh @ uh.T), 1.0)
        Aw = np.where(adj, (1 - D / Rr) * dirsim, 0.0).astype(np.float16)     # distance x heading similarity
        deg = adj.sum(1)
        # ---- CIS components, each in [0, 1]
        Cd = float(np.clip(deg.mean() / CFG["DEG_REF"], 0, 1))
        if n >= 2:
            Dn = D + np.eye(n) * 1e9
            nn_ = Dn.min(1)
            Cp = float(np.mean(np.clip(1 - nn_ / (CFG["R_FACTOR"] * h), 0, 1)))
        else:
            Cp = 0.0
        if moving.sum() >= 3:
            s_ = min(np.std(speed[moving]) / 1.0, 1.0)
            dis = 1 - float(np.linalg.norm(uh[moving].mean(0)))
            Cm = float(np.clip(0.5 * s_ + 0.5 * dis, 0, 1))
        else:
            Cm = 0.0
        CIS = wd * Cd + wp * Cp + wm * Cm
        G = np.array([Cd, Cp, Cm, CIS, min(n / CFG["N_REF"], 1), min(speed.mean() / CFG["V_REF"], 1)], np.float32)
        feats.append(dict(t=st["t"], frame=st["frame"], ids=ids, pos=P.astype(np.float32), X=X, A=Aw, G=G,
                          Cd=Cd, Cp=Cp, Cm=Cm, CIS=CIS, n=n, deg=deg))
    return feats


def band(cis):
    return "No crowd / free flow" if cis < 0.30 else ("Congested / watch" if cis < CFG["CIS_THRESHOLD"] else "High risk")


def label_at(key, t_sec):
    for a, b_, l in SEGMENT_LABELS.get(key, []):
        if a <= t_sec < b_:
            return l
    return VIDEO_LABELS[key]

# %% ------------------------------------------------------------------------------------------
# 5. WINDOWS  (past K graphs  ->  risk label HORIZON_S seconds later)
# ---------------------------------------------------------------------------------------------
def horizon_steps(dt, T):
    K = CFG["K_HISTORY"]
    hs = max(1, int(round(CFG["HORIZON_S"] / dt)))
    if T < K + hs + 10:
        new = max(1, T - K - 10)
        print(f"  [warn] video has only {T} steps; horizon reduced from {hs} to {new} steps")
        hs = new
    return hs


def make_windows(key, feats, dt, stride):
    K, T = CFG["K_HISTORY"], len(feats)
    hs = horizon_steps(dt, T)
    wins = []
    for s in range(0, T - K - hs + 1, stride):
        now, tgt = s + K - 1, s + K - 1 + hs
        wins.append(dict(key=key, s=s, now=now, tgt=tgt, y=int(label_at(key, feats[tgt]["t"]))))
    return wins


def collate(FE, wins, device):
    K = CFG["K_HISTORY"]; B = len(wins)
    N = max(1, max(len(FE[w["key"]][t]["ids"]) for w in wins for t in range(w["s"], w["s"] + K)))
    X = np.zeros((B, K, N, NODE_F), np.float32); A = np.zeros((B, K, N, N), np.float32)
    M = np.zeros((B, K, N), np.float32); G = np.zeros((B, K, GLOB_F), np.float32)
    for b, w in enumerate(wins):
        for k in range(K):
            f = FE[w["key"]][w["s"] + k]; n = len(f["ids"])
            G[b, k] = f["G"]
            if n:
                X[b, k, :n] = f["X"]; A[b, k, :n, :n] = f["A"].astype(np.float32); M[b, k, :n] = 1
                A[b, k, np.arange(n), np.arange(n)] = 1.0                      # self loops
    y = np.array([w["y"] for w in wins], np.float32)
    t = lambda a: torch.from_numpy(a).to(device)
    return t(X), t(A), t(M), t(G), t(y)


# %% ------------------------------------------------------------------------------------------
# 6. ST-GNN  (spatial graph convolution per step + temporal convolution over steps)
# ---------------------------------------------------------------------------------------------
class GCNLayer(nn.Module):
    """H(l+1) = sigma( A_hat H(l) W(l) ),  A_hat = D^-1/2 (A+I) D^-1/2  (weighted adjacency)."""
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o)

    def forward(self, x, A):
        d = A.sum(-1).clamp(min=1e-6).pow(-0.5)
        return (d.unsqueeze(-1) * A * d.unsqueeze(-2)) @ self.lin(x)


class STGNN(nn.Module):
    def __init__(self, mode="full", hidden=32):
        super().__init__()
        self.mode, self.h = mode, hidden
        self.g1, self.g2 = GCNLayer(NODE_F, hidden), GCNLayer(hidden, hidden)
        self.tconv = nn.Sequential(nn.Conv1d(2 * hidden + GLOB_F, 64, 3, padding=1), nn.ReLU(),
                                   nn.Conv1d(64, 64, 3, padding=1), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, X, A, M, G):
        if self.mode == "no_temporal":                       # ablation: only the latest graph
            X, A, M, G = X[:, -1:], A[:, -1:], M[:, -1:], G[:, -1:]
        B, K, N, _ = X.shape
        if self.mode == "no_graph":                           # ablation: no graph branch
            z = torch.cat([torch.zeros(B, K, 2 * self.h, device=X.device), G], -1)
        else:
            x, a = X.reshape(B * K, N, NODE_F), A.reshape(B * K, N, N)
            m = M.reshape(B * K, N, 1)
            h = torch.relu(self.g1(x, a)); h = torch.relu(self.g2(h, a)) * m
            mean = h.sum(1) / m.sum(1).clamp(min=1); mx = h.max(1).values
            z = torch.cat([mean, mx], -1).reshape(B, K, -1)
            z = torch.cat([z, G], -1)
        t = self.tconv(z.transpose(1, 2))
        return self.head(torch.cat([t.mean(-1), t.max(-1).values], -1)).squeeze(-1)   # logit


@torch.no_grad()
def predict(model, FE, wins, bs=64):
    model.eval(); out = []
    for i in range(0, len(wins), bs):
        X, A, M, G, _ = collate(FE, wins[i:i + bs], DEVICE)
        out.append(torch.sigmoid(model(X, A, M, G)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


def pick_threshold(y, p):
    best, ts = -1, []
    for t in np.arange(0.05, 0.96, 0.01):
        f = f1_score(y, (p >= t).astype(int), zero_division=0)
        if f > best + 1e-9:
            best, ts = f, [t]
        elif abs(f - best) <= 1e-9:
            ts.append(t)
    return float(np.mean(ts)) if ts else 0.5


def train_model(FE, tr_w, va_w, mode, seed):
    seed_all(seed)
    model = STGNN(mode, CFG["HIDDEN"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
    ytr = np.array([w["y"] for w in tr_w]); pos = max(1, int(ytr.sum())); neg = max(1, len(ytr) - int(ytr.sum()))
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / pos, device=DEVICE))
    best, best_state, bad = 1e9, None, 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); idx = np.random.permutation(len(tr_w))
        for i in range(0, len(idx), CFG["BATCH"]):
            X, A, M, G, y = collate(FE, [tr_w[j] for j in idx[i:i + CFG["BATCH"]]], DEVICE)
            opt.zero_grad(); loss = lossf(model(X, A, M, G), y); loss.backward(); opt.step()
        model.eval(); vl = 0.0
        with torch.no_grad():
            for i in range(0, len(va_w), 64):
                X, A, M, G, y = collate(FE, va_w[i:i + 64], DEVICE)
                vl += lossf(model(X, A, M, G), y).item() * len(y)
        vl /= max(1, len(va_w))
        if vl < best - 1e-4:
            best, bad = vl, 0; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]:
                break
    model.load_state_dict(best_state)
    pv = predict(model, FE, va_w)
    thr = pick_threshold(np.array([w["y"] for w in va_w]), pv)
    return model, thr


def metrics_of(y, pred, prob=None):
    y, pred = np.asarray(y), np.asarray(pred)
    cm = confusion_matrix(y, pred, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()
    m = dict(accuracy=accuracy_score(y, pred), precision=precision_score(y, pred, zero_division=0),
             recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0),
             balanced_accuracy=balanced_accuracy_score(y, pred), mcc=matthews_corrcoef(y, pred),
             tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), n=int(len(y)))
    if prob is not None and len(np.unique(y)) == 2:
        m["roc_auc"] = roc_auc_score(y, prob)
    return m


# %% ------------------------------------------------------------------------------------------
# 7. LEAVE-ONE-VIDEO-OUT EVALUATION  (test video is NEVER seen in training)
# ---------------------------------------------------------------------------------------------
def run_lovo(FE, DT, keys, mode):
    res, models = {}, {}
    K = CFG["K_HISTORY"]
    for test_key in keys:
        tr_w, va_w = [], []
        for k in keys:
            if k == test_key:
                continue
            w = make_windows(k, FE[k], DT[k], CFG["WIN_STRIDE_TRAIN"])
            if not w:
                continue
            cut = int(len(w) * (1 - CFG["VAL_FRAC"]))
            gap = (K + horizon_steps(DT[k], len(FE[k]))) // CFG["WIN_STRIDE_TRAIN"] + 1       # purge overlap
            tr_w += w[:max(0, cut - gap)]; va_w += w[cut:]
        if not tr_w or not va_w:
            print("  skipping fold", test_key); continue
        t0 = time.time()
        model, thr = train_model(FE, tr_w, va_w, mode, CFG["SEED"])
        te_w = make_windows(test_key, FE[test_key], DT[test_key], 1)
        p = predict(model, FE, te_w)
        res[test_key] = dict(wins=te_w, prob=p, y=np.array([w["y"] for w in te_w]), thr=thr)
        models[test_key] = model
        m = metrics_of(res[test_key]["y"], (p >= thr).astype(int))
        print(f"    [{mode}] test={test_key} thr={thr:.2f} acc={m['accuracy']:.3f} "
              f"P={m['precision']:.3f} R={m['recall']:.3f} ({time.time() - t0:.0f}s)")
    return res, models


def pooled(res, use_thr=True):
    y = np.concatenate([r["y"] for r in res.values()]); p = np.concatenate([r["prob"] for r in res.values()])
    pred = np.concatenate([(r["prob"] >= (r["thr"] if use_thr else 0.5)).astype(int) for r in res.values()])
    return y, p, pred, metrics_of(y, pred, p)


def reactive_baseline(FE, res, thr=None):
    """Reactive, CrowdSentry-style: alert if the CURRENT frame's CIS >= 0.50 (no learning, no forecast)."""
    thr = CFG["CIS_THRESHOLD"] if thr is None else thr
    y, pred, sc = [], [], []
    for k, r in res.items():
        for w, yy in zip(r["wins"], r["y"]):
            c = FE[k][w["now"]]["CIS"]; y.append(yy); sc.append(c); pred.append(int(c >= thr))
    return metrics_of(y, pred, sc)


def video_level(res):
    rows = []
    for k, r in res.items():
        pred = (r["prob"] >= r["thr"]).astype(int)
        rows.append(dict(video=k, true_label=int(round(r["y"].mean())), windows=len(pred),
                         frac_predicted_risky=float(pred.mean()), mean_forecast=float(r["prob"].mean()),
                         per_video_accuracy=float((pred == r["y"]).mean()),
                         video_decision=int(pred.mean() >= 0.5)))
    return pd.DataFrame(rows)


# %% ------------------------------------------------------------------------------------------
# 8. EXPLAINABILITY  (gradient x input attribution on nodes, edges and global descriptors)
# ---------------------------------------------------------------------------------------------
FACTOR_GROUPS = {
    "Sudden velocity change": ("node", [2, 3]),
    "Opposite / changing direction": ("node", [4, 5]),
    "Position in frame": ("node", [0, 1]),
    "High local density": ("glob", [0]),
    "Close proximity": ("glob", [1]),
    "Movement disorder": ("glob", [2]),
    "Instantaneous CIS": ("glob", [3]),
    "Crowd size": ("glob", [4]),
    "Mean crowd speed": ("glob", [5]),
}


def explain_window(model, FE, w):
    model.eval()
    X, A, M, G, _ = collate(FE, [w], DEVICE)
    X.requires_grad_(True); A.requires_grad_(True); G.requires_grad_(True)
    logit = model(X, A, M, G); logit.backward()
    aX = (X * X.grad).abs().detach().cpu().numpy()[0]          # [K,N,F]
    aA = (A * A.grad).abs().detach().cpu().numpy()[0]          # [K,N,N]
    aG = (G * G.grad).abs().detach().cpu().numpy()[0]          # [K,Gf]
    tot = aX.sum() + aA.sum() + aG.sum() + 1e-12
    shares = dict(node=float(aX.sum() / tot), edge=float(aA.sum() / tot), global_=float(aG.sum() / tot))
    n_last = len(FE[w["key"]][w["now"]]["ids"])            # drop the padding rows (this was the IndexError)
    node_imp = aX[-1][:n_last].sum(-1); node_imp = node_imp / (node_imp.sum() + 1e-12)
    edge_imp = (aA[-1] * (1 - np.eye(aA.shape[-1])))[:n_last, :n_last]; edge_imp = edge_imp / (edge_imp.sum() + 1e-12)
    fac = {}
    for name, (kind, ix) in FACTOR_GROUPS.items():
        fac[name] = float(aX[..., ix].sum() if kind == "node" else aG[:, ix].sum())
    s = sum(fac.values()) + 1e-12
    fac = {k: v / s for k, v in fac.items()}
    why = [k for k, _ in sorted(fac.items(), key=lambda kv: -kv[1]) if k not in ("Position in frame", "Instantaneous CIS")][:4]
    return dict(shares=shares, node_imp=node_imp, edge_imp=edge_imp, factors=fac, why=why,
                prob=float(torch.sigmoid(logit).item()), ids=FE[w["key"]][w["now"]]["ids"])


def recommend(p, thr, cis, why):
    if p >= max(thr, 0.8) and cis >= 0.7: return "Initiate emergency protocol"
    if p >= thr:
        if "Opposite / changing direction" in why[:2]: return "Consider crowd redirection"
        if "Sudden velocity change" in why[:2]: return "Alert security personnel"
        return "Restrict additional inflow"
    if cis >= 0.30 or p >= 0.6 * thr: return "Monitor crowd entry"
    return "Continue monitoring"

# %% ------------------------------------------------------------------------------------------
# 9. ALERT-LEVEL METRIC (alert precision)
# ---------------------------------------------------------------------------------------------
def alert_precision(res, persist=3):
    """An ALERT = a run of >= `persist` consecutive positive predictions (persistence rule, like the
    2-second rule in CrowdSentry). An alert is CORRECT if most of its target labels are truly risky."""
    n_alerts = n_correct = 0
    for r in res.values():
        pred = (r["prob"] >= r["thr"]).astype(int); y = r["y"]; i = 0
        while i < len(pred):
            if pred[i] == 1:
                j = i
                while j < len(pred) and pred[j] == 1:
                    j += 1
                if j - i >= persist:
                    n_alerts += 1; n_correct += int(y[i:j].mean() >= 0.5)
                i = j
            else:
                i += 1
    return (n_correct / n_alerts if n_alerts else float("nan")), n_alerts


# %% ------------------------------------------------------------------------------------------
# 10. FIGURES + DASHBOARD + ANNOTATED VIDEO
# ---------------------------------------------------------------------------------------------
FIG = lambda name: os.path.join(WORK, "figures", name)


def read_frame(tr, frame_idx):
    if tr.get("path") and os.path.exists(tr["path"]):
        cap = cv2.VideoCapture(tr["path"]); cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, fr = cap.read(); cap.release()
        if ok:
            return cv2.resize(fr, (tr["W"], tr["H"])) if fr.shape[1] != tr["W"] else fr
    return np.full((tr["H"], tr["W"], 3), 70, np.uint8)


def color_for(i):
    rng = np.random.RandomState(int(i) * 7 + 3); return tuple(int(c) for c in rng.randint(60, 255, 3))


def draw_scene(img, st, f, graph=True, imp=None, edge_top=None):
    for pid, (x1, y1, x2, y2) in zip(st["ids"], st["boxes"]):
        c = color_for(pid); cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), c, 2)
        cv2.putText(img, f"id{pid}", (int(x1), max(12, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1)
    if graph and len(f["ids"]) > 1:
        ii, jj = np.where(np.triu(f["A"].astype(np.float32) > 0, 1))
        for i, j in zip(ii, jj):
            cv2.line(img, tuple(int(v) for v in f["pos"][i]), tuple(int(v) for v in f["pos"][j]), (255, 0, 255), 1)
    return img


def save_figures(ctx):
    FE, TR, keys = ctx["FE"], ctx["TR"], ctx["keys"]
    res = ctx["res_full"]; models = ctx["models_full"]
    # ---- choose demo video / step
    vk = max(keys, key=lambda k: np.mean([f["n"] for f in FE[k]]))
    s_max = int(np.argmax([f["n"] for f in FE[vk]]))
    # Fig 2 detection
    img = draw_scene(read_frame(TR[vk], TR[vk]["steps"][s_max]["frame"]), TR[vk]["steps"][s_max], FE[vk][s_max], graph=False)
    cv2.imwrite(FIG("fig02_detection_tracking.png"), img)
    # Fig 3 tracking with trails
    s2 = min(len(FE[vk]) - 1, s_max + 4); panels = []
    for s in (s_max, s2):
        im = draw_scene(read_frame(TR[vk], TR[vk]["steps"][s]["frame"]), TR[vk]["steps"][s], FE[vk][s], graph=False)
        for pid in FE[vk][s]["ids"]:
            pts = []
            for q in range(max(0, s - 8), s + 1):
                m = np.where(FE[vk][q]["ids"] == pid)[0]
                if len(m): pts.append(tuple(int(v) for v in FE[vk][q]["pos"][m[0]]))
            for a, b in zip(pts[:-1], pts[1:]): cv2.line(im, a, b, color_for(pid), 2)
        panels.append(cv2.resize(im, (640, int(640 * im.shape[0] / im.shape[1]))))
    cv2.imwrite(FIG("fig03_tracking_trajectories.png"), np.hstack(panels))
    # Fig 4 population
    plt.figure(figsize=(9, 4))
    for k in keys:
        plt.plot([f["t"] for f in FE[k]], [f["n"] for f in FE[k]], label=f"{k} ({'risky' if label_at(k, 0) else 'stable'})")
    plt.xlabel("Time (s)"); plt.ylabel("Detected people"); plt.title("Crowd population over time"); plt.grid(alpha=.3)
    plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.savefig(FIG("fig04_population.png"), dpi=200); plt.close()
    # Fig 5 CIS + forecast
    fig, axs = plt.subplots(len(keys), 1, figsize=(10, 2.2 * len(keys)), sharex=False)
    axs = np.atleast_1d(axs)
    for ax, k in zip(axs, keys):
        t = [f["t"] for f in FE[k]]
        ax.plot(t, [f["CIS"] for f in FE[k]], label="CIS (current)", lw=1.5)
        if k in res:
            r = res[k]; tn = [FE[k][w["now"]]["t"] for w in r["wins"]]
            ax.plot(tn, r["prob"], label=f"Forecast risk (+{CFG['HORIZON_S']:.0f}s)", lw=1.5)
            ax.axhline(r["thr"], color="red", ls="--", lw=1, label="Alert threshold")
        ax.axhline(CFG["CIS_THRESHOLD"], color="gray", ls=":", lw=1)
        if label_at(k, 0) == 1 or SEGMENT_LABELS.get(k): ax.set_facecolor("#fff0f0")
        ax.set_ylim(0, 1); ax.set_ylabel(k); ax.grid(alpha=.3)
    axs[0].legend(fontsize=7, ncol=3, loc="upper right"); axs[-1].set_xlabel("Time (s)")
    plt.suptitle("Crowd Instability Score and forecast risk (pink background = labelled risky)", fontsize=10)
    plt.tight_layout(); plt.savefig(FIG("fig05_cis_over_time.png"), dpi=200); plt.close()
    # Fig 6 spatial graph
    f = FE[vk][s_max]
    plt.figure(figsize=(6, 5))
    ii, jj = np.where(np.triu(f["A"].astype(np.float32) > 0, 1))
    for i, j in zip(ii, jj): plt.plot(*zip(f["pos"][i], f["pos"][j]), color="gray", lw=.8, alpha=.7)
    sc = plt.scatter(f["pos"][:, 0], f["pos"][:, 1], c=f["deg"], cmap="coolwarm", s=60 + 25 * f["deg"], edgecolors="k")
    plt.gca().invert_yaxis(); plt.colorbar(sc, label="Node connectivity (neighbours)")
    plt.title(f"Spatial crowd graph ({vk}, t={f['t']:.1f}s, {f['n']} people)"); plt.xlabel("Image X (px)"); plt.ylabel("Image Y (px)")
    plt.tight_layout(); plt.savefig(FIG("fig06_spatial_graph.png"), dpi=200); plt.close()
    # Fig 7 threshold sweep (pooled out-of-fold forecasts)
    y, p, _, _ = pooled(res); ths = np.linspace(0.02, 0.98, 97)
    M_ = [metrics_of(y, (p >= t).astype(int)) for t in ths]
    plt.figure(figsize=(7, 4.2))
    for nm in ["accuracy", "precision", "recall", "f1"]: plt.plot(ths, [m[nm] for m in M_], label=nm.capitalize() if nm != "f1" else "F1-score")
    plt.axvline(np.mean([r["thr"] for r in res.values()]), color="k", ls="--", label="Mean validated threshold")
    plt.xlabel("Decision threshold on forecast risk"); plt.ylabel("Metric value"); plt.ylim(0, 1.02)
    plt.title("Performance vs decision threshold (leave-one-video-out)"); plt.grid(alpha=.3); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(FIG("fig07_threshold_sweep.png"), dpi=200); plt.close()
    # Fig 8 density heatmap (from real person centres)
    H_, W_ = 36, 64; acc = np.zeros((H_, W_), np.float32); W0, H0 = TR[vk]["W"], TR[vk]["H"]
    for f_ in FE[vk]:
        for x, y_ in f_["pos"]:
            acc[min(H_ - 1, int(y_ / H0 * H_)), min(W_ - 1, int(x / W0 * W_))] += 1
    acc = cv2.GaussianBlur(acc, (0, 0), 2.0); acc /= acc.max() + 1e-9
    plt.figure(figsize=(7, 4)); plt.imshow(acc, cmap="jet", origin="upper", aspect="auto")
    plt.colorbar(label="Normalised crowd density"); plt.title(f"Crowd density heatmap ({vk})")
    plt.xlabel("Spatial region X"); plt.ylabel("Spatial region Y"); plt.tight_layout()
    plt.savefig(FIG("fig08_density_heatmap.png"), dpi=200); plt.close()
    # Fig 9 + 11  (explanation + dashboard) on the highest-risk correctly-forecast window
    cand = [(k, i, r["prob"][i]) for k, r in res.items() for i, w in enumerate(r["wins"]) if w["y"] == 1 and r["prob"][i] >= r["thr"]]
    if not cand:
        cand = [(k, i, r["prob"][i]) for k, r in res.items() for i in range(len(r["wins"]))]
    dk, di, _ = max(cand, key=lambda c: c[2]); dw = res[dk]["wins"][di]; thr = res[dk]["thr"]
    ex = explain_window(models[dk], FE, dw); ctx["explanation"] = dict(video=dk, step=dw["now"], **{k: v for k, v in ex.items() if k not in ("node_imp", "edge_imp", "ids")})
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    order = np.argsort(-ex["node_imp"])[:12]
    axs[0].bar([f"id{ex['ids'][i]}" for i in order], ex["node_imp"][order]); axs[0].set_title("Top pedestrians (node importance)"); axs[0].tick_params(axis="x", rotation=60)
    fk = sorted(ex["factors"].items(), key=lambda kv: kv[1]); axs[1].barh([k for k, _ in fk], [v for _, v in fk]); axs[1].set_title("Factor importance (share)")
    axs[2].bar(["Node features", "Edges", "Global descriptors"], [ex["shares"]["node"], ex["shares"]["edge"], ex["shares"]["global_"]]); axs[2].set_title("Attribution share by source")
    for a in axs: a.grid(alpha=.3)
    plt.suptitle(f"Gradient-based explanation ({dk}, forecast risk {ex['prob']:.2f})"); plt.tight_layout()
    plt.savefig(FIG("fig09_explainability.png"), dpi=200); plt.close()
    # overlay: nodes sized by importance, top edges in red
    tr_, fe_ = TR[dk], FE[dk][dw["now"]]
    im = draw_scene(read_frame(tr_, tr_["steps"][dw["now"]]["frame"]), tr_["steps"][dw["now"]], fe_, graph=True)
    for i in order[:8]:
        if i < len(fe_["pos"]): cv2.circle(im, tuple(int(v) for v in fe_["pos"][i]), 14, (0, 0, 255), 3)
    top = np.dstack(np.unravel_index(np.argsort(-ex["edge_imp"].ravel())[:10], ex["edge_imp"].shape))[0] if ex["edge_imp"].size else []
    for i, j in top:
        if i < j and ex["edge_imp"][i, j] + ex["edge_imp"][j, i] > 0:
            cv2.line(im, tuple(int(v) for v in fe_["pos"][i]), tuple(int(v) for v in fe_["pos"][j]), (0, 0, 255), 3)
    cv2.imwrite(FIG("fig09b_explanation_overlay.png"), im)
    # Fig 10 performance + confusion matrix
    y, p, pred, m = pooled(res); cm = confusion_matrix(y, pred, labels=[0, 1])
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    names = ["Accuracy", "Precision", "Recall", "F1-score"]; vals = [m["accuracy"], m["precision"], m["recall"], m["f1"]]
    b = axs[0].bar(names, vals); axs[0].set_ylim(0, 1.1); axs[0].set_title("Forecast performance (leave-one-video-out)")
    for r_, v in zip(b, vals): axs[0].text(r_.get_x() + r_.get_width() / 2, v + .02, f"{v * 100:.1f}%", ha="center")
    axs[1].imshow(cm, cmap="Blues"); axs[1].set_xticks([0, 1], ["Stable", "Risky"]); axs[1].set_yticks([0, 1], ["Stable", "Risky"])
    axs[1].set_xlabel("Predicted"); axs[1].set_ylabel("Actual"); axs[1].set_title("Confusion matrix")
    for i in range(2):
        for j in range(2): axs[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
    plt.tight_layout(); plt.savefig(FIG("fig10_performance.png"), dpi=200); plt.close()
    # Fig 10b ablation
    if ctx.get("ablation"):
        ab = ctx["ablation"]; names = list(ab.keys()); w_ = 0.2
        plt.figure(figsize=(9, 4))
        for q, met in enumerate(["accuracy", "precision", "recall", "f1"]):
            plt.bar(np.arange(len(names)) + q * w_, [ab[n][met] for n in names], w_, label=met.capitalize() if met != "f1" else "F1")
        plt.xticks(np.arange(len(names)) + 1.5 * w_, names, rotation=10); plt.ylim(0, 1.05); plt.legend(); plt.grid(axis="y", alpha=.3)
        plt.title("Ablation and baseline comparison (pooled, leave-one-video-out)"); plt.tight_layout()
        plt.savefig(FIG("fig10b_ablation.png"), dpi=200); plt.close()
    # Fig 11 dashboard
    cis_now = fe_["CIS"]; status = "HIGH RISK" if ex["prob"] >= thr else ("WATCH" if cis_now >= .3 or ex["prob"] >= .6 * thr else "NORMAL")
    rec = recommend(ex["prob"], thr, cis_now, ex["why"])
    fig = plt.figure(figsize=(14, 7)); fig.patch.set_facecolor("white")
    fig.text(.02, .95, "StampedeGuard", fontsize=18, weight="bold")
    a1 = fig.add_axes([.02, .08, .55, .82]); a1.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a1.axis("off")
    a1.set_title(f"Live video & detection panel  |  detected: {fe_['n']} people", fontsize=10)
    a2 = fig.add_axes([.60, .50, .36, .40]); a2.set_xlim(-1.2, 1.2); a2.set_ylim(-.35, 1.2); a2.axis("off"); a2.set_aspect("equal")
    from matplotlib.patches import Wedge
    for v1, v2, c in [(0, .3, "#4caf50"), (.3, .5, "#ffc107"), (.5, 1, "#e53935")]:
        a2.add_patch(Wedge((0, 0), 1, 180 * (1 - v2), 180 * (1 - v1), width=.3, color=c))
    ang = np.pi * (1 - ex["prob"]); a2.plot([0, .85 * np.cos(ang)], [0, .85 * np.sin(ang)], "k", lw=3)
    a2.text(0, -.05, f"{ex['prob']:.2f}", ha="center", fontsize=22, weight="bold"); a2.text(0, -.3, status, ha="center", fontsize=14, weight="bold", color="#b71c1c" if status == "HIGH RISK" else "k")
    a2.set_title(f"Forecast risk (+{CFG['HORIZON_S']:.0f}s)   |   current CIS = {cis_now:.2f} ({band(cis_now)})", fontsize=10)
    a3 = fig.add_axes([.60, .08, .36, .34]); a3.axis("off")
    txt = "Why is the risk high? (XAI)\n" + "\n".join(f"  • {w}  ({ex['factors'][w] * 100:.0f}%)" for w in ex["why"]) + f"\n\nRecommendation:\n  {rec}\n\n(advisory only - a human operator decides)"
    a3.text(0, 1, txt, va="top", fontsize=11, family="DejaVu Sans", bbox=dict(boxstyle="round", fc="#f5f5f5"))
    plt.savefig(FIG("fig11_dashboard.png"), dpi=200, bbox_inches="tight"); plt.close()
    ctx["dashboard"] = dict(video=dk, status=status, recommendation=rec, why=ex["why"], forecast=ex["prob"], cis=cis_now, thr=thr)


def write_annotated_videos(ctx):
    FE, TR, res = ctx["FE"], ctx["TR"], ctx["res_full"]
    for k in ctx["keys"]:
        tr = TR[k]; out_path = os.path.join(WORK, "videos", f"{k}_annotated.mp4")
        scale = min(1.0, 960 / tr["W"]); size = (int(tr["W"] * scale), int(tr["H"] * scale))
        vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), max(1.0, 1 / tr["dt"]), size)
        prob = {}
        if k in res:
            for w, p in zip(res[k]["wins"], res[k]["prob"]): prob[w["now"]] = p
        thr = res[k]["thr"] if k in res else 0.5
        cap = cv2.VideoCapture(tr["path"]) if tr.get("path") and os.path.exists(tr["path"]) else None
        si, idx = 0, 0
        while si < len(tr["steps"]):
            st = tr["steps"][si]
            if cap is not None:
                if not cap.grab(): break
                if idx != st["frame"]: idx += 1; continue
                ok, fr = cap.retrieve()
                if not ok: break
                if fr.shape[1] != tr["W"]: fr = cv2.resize(fr, (tr["W"], tr["H"]))
            else:
                fr = np.full((tr["H"], tr["W"], 3), 70, np.uint8)
            f = FE[k][si]; fr = draw_scene(fr, st, f, graph=True)
            p = prob.get(si); cis = f["CIS"]
            col = (0, 200, 0) if cis < .3 else ((0, 200, 255) if cis < .5 else (0, 0, 255))
            cv2.rectangle(fr, (0, 0), (min(fr.shape[1], 640), 125), (0, 0, 0), -1)
            cv2.putText(fr, f"People: {f['n']}   CIS now: {cis:.2f} ({band(cis)})", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, .7, col, 2)
            if p is None:
                cv2.putText(fr, "Forecast: warming up...", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, .7, (255, 255, 255), 2)
            else:
                alert = p >= thr
                cv2.putText(fr, f"Forecast +{CFG['HORIZON_S']:.0f}s: {p:.2f}  {'ALERT' if alert else 'OK'}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 0, 255) if alert else (0, 255, 0), 2)
                cv2.putText(fr, "Action: " + recommend(p, thr, cis, ["Sudden velocity change"] if f["Cm"] > .5 else ["High local density"]), (10, 95), cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 2)
            vw.write(cv2.resize(fr, size)); si += 1; idx += 1
        vw.release()
        if cap is not None: cap.release()
    print("Annotated videos written to", os.path.join(WORK, "videos"))

# %% ------------------------------------------------------------------------------------------
# 11. TEXT FOR THE PAPER PLACEHOLDERS (auto-filled with the numbers from THIS run)
# ---------------------------------------------------------------------------------------------
def pct(x): return "n/a" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{100 * x:.2f}%"


def write_paper_text(ctx):
    TR, FE, keys, res = ctx["TR"], ctx["FE"], ctx["keys"], ctx["res_full"]
    y, p, pred, m = pooled(res); ap, n_alerts = alert_precision(res)
    base = ctx["baseline"]; ab = ctx.get("ablation", {}); ex = ctx["explanation"]; db = ctx["dashboard"]
    resol = sorted({f"{TR[k]['W']}x{TR[k]['H']}" for k in keys}); fps = sorted({round(TR[k]['fps'], 1) for k in keys})
    steps = sum(len(FE[k]) for k in keys); dur = sum(len(FE[k]) * TR[k]["dt"] for k in keys)
    ppl = np.concatenate([[f["n"] for f in FE[k]] for k in keys])
    risky = [k for k in keys if VIDEO_LABELS.get(k) == 1]; stable = [k for k in keys if VIDEO_LABELS.get(k) == 0]
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
    try:
        import ultralytics; ul = ultralytics.__version__
    except Exception: ul = "n/a"
    ms_yolo = np.mean([TR[k]["ms_yolo"] for k in keys]); ms_trk = np.mean([TR[k]["ms_track"] for k in keys])
    ms_graph, ms_model = ctx["ms_graph"], ctx["ms_model"]
    vl = ctx["video_level"]
    fine = lambda a, b: "higher" if a > b else ("lower" if a < b else "equal")
    T = f"""# Text for the yellow placeholders of the StampedeGuard paper
(Auto-generated from this run. Replace each yellow [[...]] in the paper with the matching text below.)
(Every number below comes from the run you just did. Re-run => numbers change => re-paste.)

## V-A  Dataset  (placeholder: dataset details)
The prototype was evaluated on the Kaggle Crowd-Detection-Video dataset (kaggle.com/datasets/{CFG['KAGGLE_DATASET']}). {len(keys)} videos ({', '.join(keys)}) were used, with resolution {', '.join(resol)} at {', '.join(map(str, fps))} fps. Frames were analysed every {CFG['TARGET_DT']} s, giving {steps} analysed frames ({dur:.0f} s in total; at most {CFG['MAX_STEPS']} frames per video). The number of detected people per frame ranged from {ppl.min()} to {ppl.max()} (mean {ppl.mean():.1f}). Each video was labelled by the authors as stable ({', '.join(stable) or 'none'}) or risky ({', '.join(risky) or 'none'}) [CONFIRM: you labelled these by watching the videos]. The dataset contains no real stampede, so "risky" means congested or disorderly crowd behaviour.

## V-B  YOLOv8  (placeholder: variant and confidence)
Person detection used YOLOv8 ({CFG['YOLO_WEIGHTS'].replace('.pt', '')}, Ultralytics {ul}) restricted to the COCO person class, with a confidence threshold of {CFG['YOLO_CONF']} and an input size of {CFG['YOLO_IMGSZ']} pixels.

## V-C  DeepSORT
Tracking used the deep-sort-realtime implementation of DeepSORT (max age 8 steps, 2 frames to confirm a track, MobileNet appearance embedder). Only confirmed tracks that were matched to a detection in the current frame were used.

## V-E  Graph  (placeholder: radius r, weighting function, frames per graph)
An edge connects two pedestrians i and j when their centre distance d_ij is smaller than r_ij = {CFG['R_FACTOR']} x (h_i + h_j)/2, where h is the bounding-box height in pixels; the radius therefore scales with the apparent person size and is perspective aware. Edge weights are w_ij = (1 - d_ij/r_ij) x s_ij, where s_ij = 0.5 + 0.5 cos(theta_i - theta_j) if both people are moving (speed > 0.3 body heights/s) and s_ij = 1 otherwise. Speed and acceleration are measured in body heights per second and per second squared. The heading is encoded as (cos theta, sin theta), so each node feature vector is X_i = [x, y, v, a, cos theta, sin theta] with x, y normalised by the image size. Graphs are built every {CFG['TARGET_DT']} s; the ST-GNN receives the last k = {CFG['K_HISTORY']} graphs ({CFG['K_HISTORY'] * CFG['TARGET_DT']:.1f} s of history). At most {CFG['MAX_NODES']} people per graph are kept.

## V-F  ST-GNN  (placeholder: architecture and training)
Spatial part: two graph-convolution layers (hidden size {CFG['HIDDEN']}, ReLU) with the symmetric-normalised weighted adjacency; node embeddings are mean- and max-pooled to a graph vector and concatenated with six graph-level descriptors (density, proximity and movement components, instantaneous CIS, crowd size, mean speed). Temporal part: two 1-D temporal convolution layers (kernel 3, 64 channels) over the k graph vectors, followed by mean/max pooling and a two-layer classifier with a sigmoid output (forecast risk). Training: binary cross-entropy with class weighting, Adam (learning rate {CFG['LR']}, weight decay {CFG['WEIGHT_DECAY']}), batch size {CFG['BATCH']}, up to {CFG['EPOCHS']} epochs with early stopping (patience {CFG['PATIENCE']}) on a validation set made of the last {int(CFG['VAL_FRAC'] * 100)}% of each training video (with an overlap gap). The decision threshold was chosen on this validation set. Evaluation used leave-one-video-out cross-validation, so the test video is never seen during training. The implementation is in PyTorch.

## V-G  Prediction horizon and label  (placeholder: value of Delta)
The prediction horizon is Delta = {CFG['HORIZON_S']} s. The risk label R_(t+Delta) is the manual label of the video (or labelled segment) at time t + Delta. The label is NOT computed from the CIS, so the evaluation is not circular.

## V-H  CIS
CIS = 0.40 C_d + 0.35 C_p + 0.25 C_m. C_d = mean number of neighbours per person divided by {CFG['DEG_REF']} (clipped to 1); C_p = mean over people of (1 - nearest-neighbour distance / r_ii), clipped to [0,1]; C_m = 0.5 x min(std of speed, 1) + 0.5 x (1 - mean resultant length of the headings of moving people), which is 0 for uniform flow and approaches 1 for opposing flows; it is set to 0 when fewer than 3 people move. CIS bands follow Table II. CIS is computed for the current frame (instantaneous crowd state); the ST-GNN forecast risk is the predicted probability of a risky state {CFG['HORIZON_S']} s ahead.

## V-I  Explainability  (placeholder: which method)
Explanations use gradient x input attribution of the ST-GNN output with respect to (i) node features, (ii) edge weights and (iii) the graph-level descriptors. Node importance is the normalised attribution of a pedestrian in the latest graph, edge importance is the normalised attribution of an edge, and factor importance groups the attribution of related inputs (e.g. speed and acceleration -> "sudden velocity change"). For the explained sample ({ex['video']}, step {ex['step']}), the attribution share was {ex['shares']['node'] * 100:.1f}% node features, {ex['shares']['edge'] * 100:.1f}% edges and {ex['shares']['global_'] * 100:.1f}% graph-level descriptors; the leading factors were: {', '.join(ex['why'])}. SHAP and GNNExplainer are NOT used in this implementation, so cite them only as related work (or change the Methods text to "gradient x input attribution"). Regions/"zones" are not used: the explanation is shown directly on the frame as the highlighted pedestrians and links (Fig. 9b).

## V-K  Dashboard  (placeholder: implementation stack)
The StampedeGuard dashboard in this prototype is rendered offline with OpenCV and Matplotlib from the saved results (annotated video and Fig. 11). It is not yet a web application; a React-based live front-end like the one in CrowdSentry is future work. Fig. 11 is generated from the video {db['video']} (forecast risk {db['forecast']:.2f}, status {db['status']}, recommendation: {db['recommendation']}).

## VI-A  Implementation (hardware, software, speed)
Experiments ran on {gpu} using Python {sys.version.split()[0]}, PyTorch {torch.__version__}, OpenCV {cv2.__version__}, and Ultralytics YOLOv8 {ul}. Average processing time per analysed frame: YOLOv8 {ms_yolo:.1f} ms, DeepSORT {ms_trk:.1f} ms, graph and feature construction {ms_graph:.1f} ms, ST-GNN inference {ms_model:.1f} ms per window (batch size 1).

## VI-B  Table IV (parameters)
Delta = {CFG['HORIZON_S']} s | k = {CFG['K_HISTORY']} graphs ({CFG['TARGET_DT']} s apart) | r = {CFG['R_FACTOR']} x average person height (pixels) | YOLO = {CFG['YOLO_WEIGHTS'].replace('.pt', '')}, conf {CFG['YOLO_CONF']} | CIS weights 0.40/0.35/0.25 | CIS band boundary 0.50 | analysed frame interval {CFG['TARGET_DT']} s

## VI-C  Alert precision definition
An alert is a run of at least 3 consecutive positive forecasts (persistence rule); an alert is counted as correct if most of the target labels inside the run are risky. Alert precision = correct alerts / all alerts. In this run: {pct(ap)} ({n_alerts} alerts).

## VII  Results (Table V and discussion)  -  pooled over all held-out videos, {m['n']} windows
Accuracy {pct(m['accuracy'])} | Precision {pct(m['precision'])} | Recall {pct(m['recall'])} | F1 {pct(m['f1'])} | ROC-AUC {pct(m.get('roc_auc'))} | Alert precision {pct(ap)}
Confusion matrix: TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}

Comparison (same windows, same videos):
"""
    T += f"- Reactive baseline (alert when current CIS >= 0.50): accuracy {pct(base['accuracy'])}, precision {pct(base['precision'])}, recall {pct(base['recall'])}, F1 {pct(base['f1'])}.\n"
    T += f"- Full ST-GNN forecast: accuracy {pct(m['accuracy'])}, precision {pct(m['precision'])}, recall {pct(m['recall'])}, F1 {pct(m['f1'])} -> F1 is {fine(m['f1'], base['f1'])} than the reactive baseline.\n"
    for nm, v in ab.items():
        T += f"- {nm}: accuracy {pct(v['accuracy'])}, precision {pct(v['precision'])}, recall {pct(v['recall'])}, F1 {pct(v['f1'])}.\n"
    T += "\nVideo-level view (one decision per video, only 7 videos, so treat with caution):\n" + vl.to_string(index=False) + "\n"
    T += f"""
## Notes to write in the Discussion / Limitations (true for this experiment)
- Labels are video-level (or segment-level if you set SEGMENT_LABELS); with video-level labels the task partly becomes recognising what kind of crowd a video shows, so the numbers may overstate real forecasting skill.
- Only {len(keys)} videos; leave-one-video-out removes frame leakage but the test set is still tiny. No real stampede is present.
- If any metric is low, say so plainly and report it; do not tune the threshold on the test videos.
"""
    with open(os.path.join(WORK, "results", "paper_placeholder_text.md"), "w") as f:
        f.write(T)
    return T


def split_labelled(vids):
    use = [v for v in vids if VIDEO_LABELS.get(video_key(v)) is not None or SEGMENT_LABELS.get(video_key(v))]
    return use, [v for v in vids if v not in use]


def label_helper(vids, keys, stop=True):
    """Makes a preview sheet so you can watch/judge each video, then prints the dict to fill in."""
    rows = []
    for p, k in zip(vids, keys):
        cap = cv2.VideoCapture(p); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
        tiles = []
        for q in (0.1, 0.35, 0.6, 0.85):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * q)); ok, fr = cap.read()
            fr = cv2.resize(fr, (400, 225)) if ok else np.zeros((225, 400, 3), np.uint8)
            tiles.append(fr)
        cap.release(); row = np.hstack(tiles)
        cv2.putText(row, f"{k}  ({n / fps:.0f}s)", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 255, 255), 2)
        rows.append(row)
    sheet = np.vstack(rows); path = os.path.join(WORK, "figures", "label_helper.jpg"); cv2.imwrite(path, sheet)
    print("\n" + "=" * 70)
    print("STOP - NOTHING WAS ANALYSED: need at least 3 LABELLED videos, with both a 0 and a 1." if stop else "These unlabelled videos were skipped. Preview sheet:")
    print("A preview sheet was saved to:", path)
    try:
        from IPython.display import display, Image as _Img; display(_Img(path))
    except Exception:
        pass
    print("\nWatch the full videos, then edit VIDEO_LABELS near the top of first.py, e.g.:")
    print("VIDEO_LABELS = {")
    for k in keys: print(f'    "{k}": 0,   # <- change to 1 if the crowd is congested / disorderly / risky')
    print("}")
    print("=" * 70)


# %% ------------------------------------------------------------------------------------------
# 12. MAIN
# ---------------------------------------------------------------------------------------------
class _Tee:
    def __init__(self, path): self.f = open(path, "w"); self.o = sys.stdout
    def write(self, s): self.o.write(s); self.f.write(s)
    def flush(self): self.o.flush(); self.f.flush()


def main():
    sys.stdout = _Tee(os.path.join(WORK, "results", "run_log.txt"))
    t_start = time.time(); TR = {}
    print("Labels:", LABEL_SOURCE, VIDEO_LABELS)
    if SYNTH:
        for i, (k, l) in enumerate(VIDEO_LABELS.items()):
            if i < 6: TR[k] = synthetic_tracks(k, l, i)
    else:
        from ultralytics import YOLO
        vids = get_videos(); print("\nVideos found:"); [print("  ", v) for v in vids]
        use, skipped = split_labelled(vids)
        if skipped:
            print("\n[INFO] These videos have NO label in VIDEO_LABELS and will be SKIPPED:", [video_key(v) for v in skipped])
            label_helper(skipped, [video_key(v) for v in skipped], stop=False)
        labs_ = [VIDEO_LABELS.get(video_key(v)) for v in use]
        if len(use) < 3 or len(set(labs_)) < 2:
            label_helper(vids, [video_key(v) for v in vids], stop=True)
            return
        vids = use
        print("\nLoading YOLOv8 weights ..."); yolo = YOLO(CFG["YOLO_WEIGHTS"]); print("YOLOv8 + DeepSORT tracking (cached after the first run) ...")
        for p in vids:
            TR[video_key(p)] = track_video(p, video_key(p), yolo)
    t0 = time.time(); FE = {k: build_features(tr) for k, tr in TR.items()}
    ms_graph = 1000 * (time.time() - t0) / max(1, sum(len(v) for v in FE.values()))
    DT = {k: tr["dt"] for k, tr in TR.items()}
    dropped = [k for k in TR if len(FE[k]) < CFG["K_HISTORY"] + 12]
    if dropped: print(f"  [warn] too short to use (need >= {CFG['K_HISTORY'] + 12} analysed frames): {dropped}")
    keys = [k for k in TR if len(FE[k]) >= CFG["K_HISTORY"] + 12]
    labs = [VIDEO_LABELS[k] for k in keys]
    print(f"\nVideos used: {keys}  labels: {labs}")
    if len(set(labs)) < 2 or len(keys) < 3:
        raise RuntimeError("Need at least 3 labelled videos including both classes.")


    # ---- DATA DIAGNOSTICS: do the "risky" videos really look different from the "stable" ones?
    rows = []
    for k in keys:
        F_ = FE[k]; nw = len(make_windows(k, F_, DT[k], 1))
        rows.append(dict(video=k, label=VIDEO_LABELS.get(k), seconds=round(len(F_) * DT[k]), steps=len(F_), windows=nw,
                         people=round(np.mean([f["n"] for f in F_]), 1), density_C_d=round(np.mean([f["Cd"] for f in F_]), 3),
                         proximity_C_p=round(np.mean([f["Cp"] for f in F_]), 3), movement_C_m=round(np.mean([f["Cm"] for f in F_]), 3),
                         CIS=round(np.mean([f["CIS"] for f in F_]), 3)))
    diag = pd.DataFrame(rows); print("\nDATA DIAGNOSTICS (per video):"); print(diag.to_string(index=False))
    diag.to_csv(os.path.join(WORK, "results", "data_diagnostics.csv"), index=False)
    if diag["windows"].sum() < 600:
        print(f"\n[WARNING] Only {int(diag['windows'].sum())} training windows in total. That is very little for a neural network;"
              " results will be unstable and should NOT be reported as a performance claim. Use more / longer videos or segment labels.")

    print("\nTraining ST-GNN (leave-one-video-out) ...")
    res_full, models_full = run_lovo(FE, DT, keys, "full")
    keys = [k for k in keys if k in res_full]
    ctx = dict(TR=TR, FE=FE, keys=keys, res_full=res_full, models_full=models_full, ms_graph=ms_graph)
    ctx["baseline"] = reactive_baseline(FE, res_full)
    yall = pooled(res_full)[0]
    ctx["ablation"] = {"Trivial: always 'risky'": metrics_of(yall, np.ones_like(yall)),
                       "Trivial: always 'stable'": metrics_of(yall, np.zeros_like(yall)),
                       "Reactive CIS>=0.5 (baseline)": ctx["baseline"], "Full ST-GNN": pooled(res_full)[3]}
    if CFG["RUN_ABLATIONS"]:
        for mode, name in [("no_graph", "Ablation: no graph (descriptors only)"), ("no_temporal", "Ablation: no temporal (last graph only)")]:
            print(f"\nAblation: {mode}"); r_, _ = run_lovo(FE, DT, keys, mode)
            if r_: ctx["ablation"][name] = pooled(r_)[3]
    # timing of one inference
    k0 = keys[0]; w0 = res_full[k0]["wins"][:20]; mdl = models_full[k0]; t0 = time.time()
    for w in w0: predict(mdl, FE, [w])
    ctx["ms_model"] = 1000 * (time.time() - t0) / max(1, len(w0))
    ctx["video_level"] = video_level(res_full)

    print("\nCreating figures ...")
    save_figures(ctx)
    if CFG["WRITE_VIDEOS"]:
        write_annotated_videos(ctx)

    y, p, pred, m = pooled(res_full); ap, na = alert_precision(res_full)
    print("\n" + "=" * 70); print("FINAL RESULTS (pooled, leave-one-video-out, unseen test videos)"); print("=" * 70)
    print(f"Accuracy  : {pct(m['accuracy'])}\nPrecision : {pct(m['precision'])}\nRecall    : {pct(m['recall'])}\n"
          f"F1-score  : {pct(m['f1'])}\nROC-AUC   : {pct(m.get('roc_auc'))}\nAlert prec: {pct(ap)}  ({na} alerts)")
    print(f"Confusion : TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}")
    n_pred_classes = len(np.unique(pred)); auc = m.get("roc_auc", float("nan"))
    valid = (n_pred_classes == 2) and (auc == auc) and auc >= 0.6 and m["balanced_accuracy"] >= 0.6
    print(f"Balanced accuracy: {pct(m['balanced_accuracy'])}   MCC: {m['mcc']:.3f}")
    if not valid:
        print("\n[WARNING] NOT A VALID PERFORMANCE CLAIM: the model predicts only one class and/or ROC-AUC < 0.6 and/or balanced accuracy < 60%."
              "\n          Accuracy/precision/recall here just reflect the class balance. Do NOT report them as results; fix data/labels/detection first.")
    ctx["valid_claim"] = bool(valid)
    print("\nComparison:"); [print(f"  {n:42s} acc={pct(v['accuracy'])} P={pct(v['precision'])} R={pct(v['recall'])} F1={pct(v['f1'])}") for n, v in ctx["ablation"].items()]
    print("\nPer-video:"); print(ctx["video_level"].to_string(index=False))

    report = [["Metric", "Value"], ["Accuracy", pct(m["accuracy"])], ["Precision", pct(m["precision"])], ["Recall", pct(m["recall"])],
              ["F1-score", pct(m["f1"])], ["ROC-AUC", pct(m.get("roc_auc"))], ["Balanced accuracy", pct(m["balanced_accuracy"])],
              ["Alert precision", pct(ap)], ["Alerts", na], ["TN", m["tn"]], ["FP", m["fp"]], ["FN", m["fn"]], ["TP", m["tp"]],
              ["Valid performance claim", bool(valid)]]
    pd.DataFrame(report[1:], columns=report[0]).to_csv(os.path.join(WORK, "results", "performance_summary.csv"), index=False)
    pd.DataFrame(ctx["ablation"]).T.to_csv(os.path.join(WORK, "results", "model_comparison.csv"))
    with open(os.path.join(WORK, "results", "ACCURACY_REPORT.txt"), "w") as fh:
        fh.write("STAMPEDEGUARD - PERFORMANCE REPORT (leave-one-video-out, unseen test videos)\n" + "=" * 64 + "\n")
        for r_ in report[1:]: fh.write(f"{r_[0]:26s}: {r_[1]}\n")
        fh.write("\nComparison with baselines / ablations\n")
        for n_, v_ in ctx["ablation"].items():
            fh.write(f"  {n_:42s} acc={pct(v_['accuracy'])} P={pct(v_['precision'])} R={pct(v_['recall'])} F1={pct(v_['f1'])}\n")
        fh.write("\nPer-video results\n" + ctx["video_level"].to_string(index=False) + "\n")
    json.dump(dict(valid_performance_claim=bool(valid), pooled=m, alert_precision=ap, n_alerts=na, comparison=ctx["ablation"], explanation=ctx["explanation"],
                   dashboard=ctx["dashboard"], config={k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
                   labels=VIDEO_LABELS, runtime_s=time.time() - t_start),
              open(os.path.join(WORK, "results", "metrics.json"), "w"), indent=2, default=float)
    ctx["video_level"].to_csv(os.path.join(WORK, "results", "video_level_results.csv"), index=False)
    rows = [dict(video=k, t=FE[k][w["now"]]["t"], cis_now=FE[k][w["now"]]["CIS"], forecast=float(pr), thr=r["thr"], label=int(w["y"]))
            for k, r in res_full.items() for w, pr in zip(r["wins"], r["prob"])]
    pd.DataFrame(rows).to_csv(os.path.join(WORK, "results", "window_predictions.csv"), index=False)
    write_paper_text(ctx)
    print("\nPaper text for the yellow placeholders ->", os.path.join(WORK, "results", "paper_placeholder_text.md"))
    zp = shutil.make_archive(os.path.join(os.path.dirname(WORK), "stampedeguard_outputs"), "zip", WORK)
    print("Everything zipped ->", zp)
    try:
        from google.colab import files; files.download(zp)
    except Exception:
        pass
    print(f"\nDONE in {(time.time() - t_start) / 60:.1f} min")


if __name__ == "__main__":
    main()

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarni

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>